# 14 Small additions: q-values, automatic ARIMA, post-hoc tests, lines, Box-Cox, MANOVA

Words and options added to simplydata after 0.2.0 that notebooks 01 to 07 do not cover, each on published data:

- `table.add_adjusted_p_value_column(method="storey" | "local fdr")`: Storey's q-values and the local false
  discovery rate for the 3170 genes of Hedenfalk et al. (2001), whose pi0 Storey & Tibshirani (2003) printed (0.67).
- `table.forecast(method="arima", order="automatic")` on the log of the airline passengers (Box & Jenkins 1976,
  series G): the order must be their airline model ARIMA(0,1,1)(0,1,1)12; the estimates and forecasts against R and
  statsmodels.
- `table.compare_each_pair_of(matched_by=, test="nemenyi" | "conover")` after Friedman's test, on Hollander & Wolfe's
  (1973) rounding-first-base times; `compare_each_pair_of(based_on="percents")` on Fleiss's (1981) smokers.
- `table.fit_dose_response` (the four-parameter log-logistic curve) on the ryegrass data (Inderjit et al. 2002).
- `table.find_best_power_of` (Box-Cox) on Box & Cox's (1964) poison survival times, whose power they printed (-0.75).
- `table.fit_line(method=)`: standardised major axis (Wright et al. leaf lifespans, Warton et al. 2006), Deming and
  Passing-Bablok (the mcr creatinine data), Theil-Sen (Anscombe's set 3; creatinine).
- `table.find_correlation_between(show="covariance")` on mtcars, and `table.compare_averages_of([...])` (MANOVA) on
  iris.

**Only simplydata runs in this notebook.** The reference values (`references/14_small_additions.csv`, from
`scripts/references/14_small_additions.py`), each printed with its source: the published values (half a unit of the last printed digit); the Python parent where there is one
(statsmodels, SciPy, pandas); the formula written out in NumPy/SciPy where one exists (an independent calculation);
and R, run here (qvalue, forecast, stats, PMCMRplus, drc, car, smatr, mcr, deming).

Tolerances (the same throughout the notebook): relative 1e-10 for closed formulas (q-values given pi0, pairwise
tests, the standardised major axis and Deming slopes, covariances, MANOVA statistics), relative 1e-6 for anything
fitted by iteration (ARIMA, the dose-response curve, the Box-Cox power and its interval).

The tables read here are the data files of this repository with the columns the notebook needs (the log of the
passengers, the rounding times one row per player and route, one row per patient for the smokers, the log10 leaf
measures), written by the reference script to `references/data/`.

In [1]:
import re

import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_numbers, read_row, read_rows, versions

NOTEBOOK = "14_small_additions"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def rows(word, *args, **options):
    return read_rows(quiet(word, *args, **options))


def keyed(word, column, *args, **options):
    return {row[column]: row for row in rows(word, *args, **options)}

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Storey's q-values and the local false discovery rate: `add_adjusted_p_value_column`

Storey & Tibshirani (2003) estimated that 67 % of the 3170 Hedenfalk genes are not differentially expressed
(pi0 = 0.67). simplydata gives the q-values, not pi0 itself; pi0 is read back as the q-value of the largest p-value
divided by that p-value (that q-value is pi0 times it). Given pi0, the q-values follow from a closed formula,
q_(i) = min over j >= i of pi0 m p_(j) / j, written out (NumPy, in the reference script) with R qvalue's pi0. R: `qvalue::qvalue(p)` with
its defaults (pi0 by the smoother; local FDR from `lfdr()`).

In [2]:
hedenfalk = load("hedenfalk")
quiet(hedenfalk.add_adjusted_p_value_column, from_column="p", method="storey").show_first_rows()

,p,p (Storey q-value)
1,0.012126,0.088192
2,0.075025,0.209367
3,0.994921,0.667999
4,0.041785,0.161636
5,0.845814,0.632474


In [3]:
dataset = "Hedenfalk genes (qvalue)"
p = read_numbers(hedenfalk.get_column("p"))
q = read_numbers(quiet(load("hedenfalk").add_adjusted_p_value_column, from_column="p",
                       method="storey").get_column("p (Storey q-value)"))
lfdr = read_numbers(quiet(load("hedenfalk").add_adjusted_p_value_column, from_column="p",
                          method="local fdr").get_column("local false discovery rate"))
largest = p.index(max(p))
ROWS = [0, 1, 2, p.index(min(p)), largest]
method = 'add_adjusted_p_value_column (method="storey")'
checks.check(dataset, method, "pi0", q[largest] / p[largest])
checks.check(dataset, method, "q-values below 0.05", sum(1 for v in q if v < 0.05))
for i in ROWS:
    checks.check(dataset, method, f"q-value of row {i + 1}", q[i])
checks.check(dataset, method, "all 3170 q-values", q)
method = 'add_adjusted_p_value_column (method="local fdr")'
checks.check(dataset, method, "local FDR below 0.05", sum(1 for v in lfdr if v < 0.05))
for i in ROWS:
    checks.check(dataset, method, f"local FDR of row {i + 1}", lfdr[i])
checks.check(dataset, method, "all 3170 local FDRs", lfdr)

agrees  Hedenfalk genes (qvalue) | add_adjusted_p_value_column (method="storey") | pi0: simplydata 0.669926026475, Storey & Tibshirani (2003), printed 0.67; difference 7.40e-05 (allowed 5e-03)
      Source: Storey & Tibshirani (2003), printed (printed) - https://doi.org/10.1073/pnas.1530509100
agrees  Hedenfalk genes (qvalue) | add_adjusted_p_value_column (method="storey") | pi0: simplydata 0.669926026475, R qvalue 0.669926026475; relative difference 0.00e+00 (allowed 1e-10)
      Source: R qvalue (run reference) - https://cran.r-project.org/package=qvalue (version 2.44.0)
agrees  Hedenfalk genes (qvalue) | add_adjusted_p_value_column (method="storey") | q-values below 0.05: simplydata 162, R qvalue 162; difference 0.00e+00 (allowed 0e+00)
      Source: R qvalue (run reference) - https://cran.r-project.org/package=qvalue (version 2.44.0)
agrees  Hedenfalk genes (qvalue) | add_adjusted_p_value_column (method="storey") | q-value of row 1: simplydata 0.0881916317044, R qvalue 0.0881916317

1

## The airline model: `table.forecast(method="arima", order="automatic")`

Box & Jenkins (1976) fitted ARIMA(0,1,1)(0,1,1)12 to the log of the monthly airline passengers 1949-1960, the
"airline model". The automatic order must be this one (compared exactly). Its two estimates (ma1, sma1) and the
forecasts for 1961 are compared with R `forecast::auto.arima(stepwise = FALSE, approximation = FALSE)` and
`forecast()` (what the automatic order follows), with R `stats::arima` on the fixed order (exact maximum
likelihood) and with statsmodels `SARIMAX` on the fixed order, set as R does it: the likelihood of the
differenced series (`simple_differencing=True`; SARIMAX's default instead treats the first values with a diffuse
start, a different likelihood) and standard errors from the observed information (`cov_type="approx"`, the numerical
Hessian, as R's `var.coef`; SARIMAX's default is the outer product of gradients). Box & Jenkins estimated the two values by another
method, so their printed estimates are not used.

In [4]:
airline = load("air")
airline.turn_into_dates("month")
airline.forecast("log_passengers", sorted_by="month", method="arima", order="automatic", repeats_every=12)

month is now dates.


For "log_passengers", auto.arima chose ARIMA(0,1,1)(0,1,1)[12]: the smallest AICc among the orders it tried (stepwise = FALSE, approximation = FALSE). The order was chosen from the data, so the intervals are a little too narrow.
Forecast of "log_passengers" for the next 12 months by ARIMA(0,1,1)(0,1,1)[12] (1 difference and 1 past error; for the season, 1 difference 12 rows back and 1 past error 12 rows back): estimates ma1 -0.402, sma1 -0.557. Next month: 6.11 (95 % prediction interval 6.04 to 6.18); in 12 months: 6.17 (6.01 to 6.33). The leftover one-step errors look like noise (Ljung-Box up to 24 rows back, p = 0.233).
Method: ARIMA(0,1,1)(0,1,1)[12], as R forecast::auto.arima(x, stepwise = FALSE, approximation = FALSE) and forecast::forecast(fit, h); 95 % intervals (run in R).


,month,steps ahead,forecast,forecast 95 % prediction low,forecast 95 % prediction high,method
1,1961-01-01,1,6.110186,6.037607,6.182764,"ARIMA(0,1,1)(0,1,1)[12]"
2,1961-02-01,2,6.053775,5.969203,6.138347,"ARIMA(0,1,1)(0,1,1)[12]"
3,1961-03-01,3,6.171715,6.076650,6.266779,"ARIMA(0,1,1)(0,1,1)[12]"
4,1961-04-01,4,6.199300,6.094792,6.303809,"ARIMA(0,1,1)(0,1,1)[12]"
5,1961-05-01,5,6.232556,6.119388,6.345724,"ARIMA(0,1,1)(0,1,1)[12]"
6,1961-06-01,6,6.368779,6.247569,6.489988,"ARIMA(0,1,1)(0,1,1)[12]"
7,1961-07-01,7,6.507294,6.378544,6.636044,"ARIMA(0,1,1)(0,1,1)[12]"
8,1961-08-01,8,6.502906,6.367034,6.638779,"ARIMA(0,1,1)(0,1,1)[12]"
9,1961-09-01,9,6.324698,6.182058,6.467338,"ARIMA(0,1,1)(0,1,1)[12]"
10,1961-10-01,10,6.209008,6.059908,6.358109,"ARIMA(0,1,1)(0,1,1)[12]"


In [5]:
dataset = "AirPassengers (log)"
method = 'forecast (order="automatic")'
model = keyed(airline.forecast, "term", "log_passengers", sorted_by="month", method="arima", order="automatic",
              repeats_every=12, show="model")
ahead = rows(airline.forecast, "log_passengers", sorted_by="month", method="arima", order="automatic", repeats_every=12)
order = [int(v) for v in re.findall(r"\d+", ahead[0]["method"])[:7]]
for name, found in zip(("p", "d", "q", "seasonal p", "seasonal d", "seasonal q", "period"), order):
    checks.check(dataset, method, f"order: {name}", found)
checks.check_values(dataset, method, {
    "ma1": model["ma1"]["estimate"], "sma1": model["sma1"]["estimate"],
    "ma1 standard error": model["ma1"]["standard error"], "sma1 standard error": model["sma1"]["standard error"],
    "log-likelihood": model["log-likelihood"]["estimate"],
    "forecast 1961-01": ahead[0]["forecast"], "forecast 1961-12": ahead[11]["forecast"],
    "1961-01 95 % low": ahead[0]["forecast 95 % prediction low"],
    "1961-12 95 % high": ahead[11]["forecast 95 % prediction high"]})

agrees  AirPassengers (log) | forecast (order="automatic") | order: p: simplydata 0, Box & Jenkins (1976), the airline model 0; difference 0.00e+00 (allowed 0e+00)
      Source: Box & Jenkins (1976), the airline model (printed)
agrees  AirPassengers (log) | forecast (order="automatic") | order: p: simplydata 0, R forecast auto.arima 0; difference 0.00e+00 (allowed 0e+00)
      Source: R forecast auto.arima (run reference) - https://search.r-project.org/CRAN/refmans/forecast/html/auto.arima.html (version 9.0.2)
agrees  AirPassengers (log) | forecast (order="automatic") | order: d: simplydata 1, Box & Jenkins (1976), the airline model 1; difference 0.00e+00 (allowed 0e+00)
      Source: Box & Jenkins (1976), the airline model (printed)
agrees  AirPassengers (log) | forecast (order="automatic") | order: d: simplydata 1, R forecast auto.arima 1; difference 0.00e+00 (allowed 0e+00)
      Source: R forecast auto.arima (run reference) - https://search.r-project.org/CRAN/refmans/forecast/html/

19

## After Friedman's test: `compare_each_pair_of(matched_by=, test="nemenyi" | "conover")`

22 baseball players each ran to second base by three routes (Hollander & Wolfe 1973, p. 140; the example of R's
`friedman.test` help, whose Friedman chi-squared is 11.143, printed below as a check of the data). Within each player
the three times are ranked; Nemenyi's test compares the mean ranks of each pair with the studentized range,
q = |mean rank difference| / sqrt(k (k + 1) / (12 n)), p = P(range of k standard normals > q) (written out with SciPy
`studentized_range` in the reference script, which also prints R's Friedman chi-squared of these data). Conover's test uses the t distribution on (n - 1)(k - 1) degrees of freedom; simplydata
adjusts its p-values by Holm's method. R: PMCMRplus `frdAllPairsNemenyiTest` and `frdAllPairsConoverTest(p.adjust.method
= "holm")`.

In [6]:
long = load("rounding_long")
long.compare_each_pair_of("time", for_each_group_in="route", matched_by="player", test="nemenyi")

Note: "test=" is now called "test_method=". The old name will stop working in a later version.


Each pair of "route" groups compared on "time" within the same 22 "player" values with Nemenyi's test (ranks within each "player"), run in R as PMCMRplus::frdAllPairsNemenyiTest(): 3 pairs; Nemenyi's test allows for all the pairs itself (the studentized range), so there is no further adjustment: do not adjust these p-values again.
  narrow_angle vs round_out: narrow_angle ranks lower than round_out within each "player" (mean rank 2.14 against 2.41); about 64 of 100 studies would show this much if the two groups had the same typical "time" (in ranks) (Nemenyi p = 0.637).
  narrow_angle vs wide_angle: narrow_angle ranks higher than wide_angle within each "player" (mean rank 2.14 against 1.45); about 6 of 100 studies would show this much if the two groups had the same typical "time" (in ranks) (Nemenyi p = 0.061).
  round_out vs wide_angle: round_out ranks higher than wide_angle within each "player" (mean rank 2.41 against 1.45); about 4 of 1000 studies would show this much if the two gro

,group 1,group 2,test,mean rank in group 1,mean rank in group 2,statistic name,statistic,p-value (adjusted),rows
1,narrow_angle,round_out,Nemenyi's test,2.136364,2.409091,q,1.279204,0.637427,22
2,narrow_angle,wide_angle,Nemenyi's test,2.136364,1.454545,q,3.198011,0.061371,22
3,round_out,wide_angle,Nemenyi's test,2.409091,1.454545,q,4.477215,0.004410,22


In [7]:
dataset = "rounding first base (Hollander & Wolfe)"
for test in ("nemenyi", "conover"):
    method = f'compare_each_pair_of (matched_by=, test="{test}")'
    for row in rows(long.compare_each_pair_of, "time", for_each_group_in="route", matched_by="player", test=test):
        pair = f"{row['group 1']} vs {row['group 2']}"
        checks.check_values(dataset, method, {
            f"{pair}: mean rank in group 1": row["mean rank in group 1"],
            f"{pair}: mean rank in group 2": row["mean rank in group 2"],
            f"{pair}: q": row["statistic"], f"{pair}: p-value": row["p-value (adjusted)"],
            f"{pair}: {row['statistic name']} (size)": abs(row["statistic"])})

agrees  rounding first base (Hollander & Wolfe) | compare_each_pair_of (matched_by=, test="nemenyi") | narrow_angle vs round_out: mean rank in group 1: simplydata 2.13636363636, pandas rank (within player) 2.13636363636; relative difference 2.08e-16 (allowed 1e-10)
      Source: pandas rank (within player) (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=rank
agrees  rounding first base (Hollander & Wolfe) | compare_each_pair_of (matched_by=, test="nemenyi") | narrow_angle vs round_out: mean rank in group 2: simplydata 2.40909090909, pandas rank (within player) 2.40909090909; relative difference 1.84e-16 (allowed 1e-10)
      Source: pandas rank (within player) (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=rank
agrees  rounding first base (Hollander & Wolfe) | compare_each_pair_of (matched_by=, test="nemenyi") | narrow_angle vs round_out: q: simplydata 1.27920429813, SciPy, the formula 1.27920429813; relative diff

## Pairs of percents: `compare_each_pair_of(based_on="percents")`

Smokers among lung-cancer patients in four groups (Fleiss 1981, p. 139; the example of R's `prop.test` help). Each
pair of groups is compared by the chi-squared test with Yates' correction (the default) or Fisher's exact test, and
the p-values are adjusted by Holm's method. Parent: SciPy `chi2_contingency(correction=True)` and `fisher_exact`
for each pair, adjusted by statsmodels `multipletests(method="holm")`. R: `pairwise.prop.test` (Yates and Holm by
default), and `fisher.test` for each pair with `p.adjust(method = "holm")`.

In [8]:
patients = load("patients")
patients.turn_into_text("group")
patients.compare_each_pair_of("smoker", for_each_group_in="group", based_on="percents", value="yes")

group is now text.
"smoker" = "yes" (chosen with value=): 83 of 86 (96.5 %) in 1, 90 of 93 (96.8 %) in 2, 129 of 136 (94.9 %) in 3 and 70 of 82 (85.4 %) in 4.
Each pair of "group" groups compared on the percent "smoker" = yes with the chi-square test with Yates' continuity correction, the 2 x 2 test of compare_counts_of (SciPy chi2_contingency), as R pairwise.prop.test(), whose default is the same; the p-values are adjusted for 6 pairs with Holm's method (statsmodels multipletests), so the chance of any false alarm stays at 5 %. Each difference has Newcombe's 95 % interval (statsmodels confint_proportions_2indep), as compare_risks.
  1 vs 2: the percent in 1 is 0.263 percentage points lower (95 % CI -6.88 to 6.01); almost all of 100 studies would show this much if the two groups had the same percent "yes" (Holm-adjusted p = 1.000).
  1 vs 3: the percent in 1 is 1.66 percentage points higher (95 % CI -5.14 to 7.25); almost all of 100 studies would show this much if the two groups had th

,group 1,group 2,test,share yes in group 1,share yes in group 2,difference,difference 95 % CI low,difference 95 % CI high,statistic name,statistic,p-value,yes in group 1,count in group 1,yes in group 2,count in group 2,p-value (Holm)
1,1,2,chi-square test with Yates' correction,0.965116,0.967742,-0.002626,-0.068845,0.060062,chi-square,0.000000,1.000000,83,86,90,93,1.000000
2,1,3,chi-square test with Yates' correction,0.965116,0.948529,0.016587,-0.051435,0.072476,chi-square,0.061679,0.803862,83,86,129,136,1.000000
3,1,4,chi-square test with Yates' correction,0.965116,0.853659,0.111458,0.024241,0.206522,chi-square,5.115505,0.023713,83,86,70,82,0.118565
4,2,3,chi-square test with Yates' correction,0.967742,0.948529,0.019213,-0.044785,0.074417,chi-square,0.136514,0.711772,90,93,129,136,1.000000
5,2,4,chi-square test with Yates' correction,0.967742,0.853659,0.114083,0.029968,0.208747,chi-square,5.854620,0.015536,90,93,70,82,0.093217
6,3,4,chi-square test with Yates' correction,0.948529,0.853659,0.094871,0.015692,0.190804,chi-square,4.656177,0.030942,129,136,70,82,0.123768


In [9]:
dataset = "smokers (Fleiss 1981)"
for test in ("chi-square", "fisher"):
    method = 'compare_each_pair_of (based_on="percents", test="fisher")' if test == "fisher" else 'compare_each_pair_of (based_on="percents")'
    extra = {"test": "fisher"} if test == "fisher" else {}
    for row in rows(patients.compare_each_pair_of, "smoker", for_each_group_in="group", based_on="percents", value="yes",
                    **extra):
        pair = f"{row['group 1']} vs {row['group 2']}"
        checks.check_values(dataset, method, {f"{pair}: p-value": row["p-value"],
                                              f"{pair}: p-value (Holm)": row["p-value (Holm)"]})

agrees  smokers (Fleiss 1981) | compare_each_pair_of (based_on="percents") | 1 vs 2: p-value: simplydata 1, SciPy 1; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/stats.html
agrees  smokers (Fleiss 1981) | compare_each_pair_of (based_on="percents") | 1 vs 2: p-value (Holm): simplydata 1, SciPy + statsmodels multipletests (Holm) 1; relative difference 0.00e+00 (allowed 1e-10)
      Source: SciPy + statsmodels multipletests (Holm) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/stats.html
agrees  smokers (Fleiss 1981) | compare_each_pair_of (based_on="percents") | 1 vs 2: p-value (Holm): simplydata 1, R pairwise.prop.test 1; relative difference 0.00e+00 (allowed 1e-10)
      Source: R pairwise.prop.test (run reference) - https://search.r-project.org/R/refmans/stats/html/pairwise.prop.test.html
agrees  smokers (Fleiss 1981) | compare_each_pair_of (based_on="percents") | 1 vs 3: p-val

## The dose-response curve: `table.fit_dose_response`

Root length of perennial ryegrass at 7 concentrations of ferulic acid (Inderjit, Streibig & Olofsdotter 2002; the
example of the R package drc). The four-parameter log-logistic curve
f(x) = c + (d - c) / (1 + exp(b (log x - log e))), with the doses that give 10, 50 and 90 % of the effect and their
delta-method intervals. R: `drc::drm(fct = LL.4())` and `drc::ED(interval = "delta")` (what simplydata calls, run
here independently, once with drc's default stopping rule and once to `relTol = 1e-12`); Python: SciPy `curve_fit`
on the same curve from drc's estimates, to a tight stopping rule, for the estimates (SciPy's standard errors come from
the Gauss-Newton approximation J'J, drc's from the full Hessian of the residual sum of squares, so they are a
different estimate and are not compared).

In [10]:
ryegrass = load("ryegrass")
ryegrass.fit_dose_response(x="conc", y="root_length", doses_for_effect=[10, 50, 90])

Fitted a 4 parameter log-logistic (R LL.4) dose-response curve of "root_length" against the dose "conc" (24 rows).
The dose for 10 % of the effect (ED10) is 1.46 (95 % CI 1.07 to 1.85); the dose for 50 % of the effect (ED50) is 3.06 (95 % CI 2.67 to 3.45); the dose for 90 % of the effect (ED90) is 6.39 (95 % CI 4.63 to 8.15).
The curve: upper limit 7.79 (95 % CI 7.40 to 8.19), lower limit 0.481 (95 % CI 0.0388 to 0.924), steepness b = 2.98.
"Effect" is the share of the way the response falls with dose: from the upper limit (no dose) to the lower limit (very high doses); b is drc's steepness (how fast the response changes around the ED50 on a log dose scale; above 0 it falls, below 0 it rises).
Dose effect: If the dose truly had no effect, a change with dose this big would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001). (drc::noEffect, a likelihood-ratio test)
Method: R drc::drm(root_length ~ conc, fct = LL.4(), control = drc::drmc(relTol = 1e-15, maxIt = 10000)) (Ri

,term,estimate,standard error,statistic,p-value,95 % CI low,95 % CI high,rows
1,b (steepness),2.982229,0.465067,6.412464,2.960669e-06,2.012115,3.952342,24
2,c (lower limit),0.481410,0.212193,2.268738,3.450703e-02,0.038783,0.924036,24
3,d (upper limit),7.792962,0.188567,41.327174,3.822001e-21,7.399617,8.186307,24
4,e (ED50),3.057956,0.185731,16.464420,4.267874e-13,2.670527,3.445384,24
5,ED10,1.463710,0.186772,NaN,NaN,1.074111,1.853309,24
6,ED50,3.057956,0.185731,NaN,NaN,2.670527,3.445384,24
7,ED90,6.388627,0.845098,NaN,NaN,4.625784,8.151470,24


In [11]:
dataset, method = "ryegrass (drc)", "fit_dose_response"
fit = keyed(ryegrass.fit_dose_response, "term", x="conc", y="root_length", doses_for_effect=[10, 50, 90])
TERMS = {"b": "b (steepness)", "c": "c (lower limit)", "d": "d (upper limit)", "e": "e (ED50)"}
for short, term in TERMS.items():
    checks.check_values(dataset, method, {f"{short} estimate": fit[term]["estimate"],
                                          f"{short} standard error": fit[term]["standard error"]})
for ed in ("ED10", "ED50", "ED90"):
    checks.check_values(dataset, method, {f"{ed} {quantity}": fit[ed][quantity]
                                          for quantity in ("estimate", "standard error", "95 % CI low", "95 % CI high")})

explained difference – precision limit (C30)  ryegrass (drc) | fit_dose_response | b estimate: simplydata 2.98222857655, R drc drm (default stopping rule) 2.98221907128; relative difference 3.19e-06 (allowed 1e-06)
      why: R drc stops short of the least-squares optimum even at relTol 1e-15; simplydata gives exactly what drc gives - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c30-r-drc-stops-short-of-the-least-squares-optimum-even-at-reltol-1e-15-simplydata-gives-exactly-what-drc-gives
      Source: R drc drm (default stopping rule) (run reference) - https://search.r-project.org/CRAN/refmans/drc/html/drm.html (version 4.0.0)
explained difference – precision limit (C30)  ryegrass (drc) | fit_dose_response | b estimate: simplydata 2.98222857655, R drc drm (relTol 1e-12) 2.98223211031; relative difference 1.18e-06 (allowed 1e-06)
      why: R drc stops short of the least-squares optimum even at relTol 1e-15; simplydata gives exactly what drc gives - https://g

## The Box-Cox power: `table.find_best_power_of`

Survival times of 48 animals given 3 poisons and 4 treatments (Box & Cox 1964, the boot package's `poisons`). For
the two-way model Box and Cox printed a best power of about -0.75 and suggested the reciprocal (power -1). The
power maximises the profile log-likelihood -n/2 log(RSS(lambda) / n) + (lambda - 1) sum(log y) of the linear model on
the Box-Cox transformed times. simplydata gives car's interval, the power plus or minus 1.96 standard errors, the
standard error from the curvature of the profile at its top (car's `summary` "Lower Bnd" and "Upper Bnd"); it is
not the profile-likelihood interval. Written out (in the reference script) with statsmodels OLS and SciPy (`minimize_scalar`; the
curvature by central differences with steps 1e-3 and 5e-4, combined by Richardson extrapolation); R `car::powerTransform` (what simplydata calls, run here).
For the times on their own the parent is SciPy `stats.boxcox(alpha=0.05)`. The poison and treatment columns are
read as text (they are labels).

In [12]:
poisons = load("poisons")
poisons.turn_into_text("poison")
poisons.turn_into_text("treatment")
poisons.find_best_power_of("time", using=["poison", "treatment"])

poison is now text.
treatment is already text.
Best power (Box-Cox): -0.750 (95 % CI -1.14 to -0.363): the power that brings the leftovers of the model explain("time", using=['poison', 'treatment']) closest to a normal shape.
A nearby simple choice: 1 / time (power -1).
  Log (power 0) is outside the interval.
  No change (power 1) is outside the interval.
Log: If the log (power 0) were truly right, a best power this far from 0 would turn up in fewer than 1 in 1000 studies like this one (p = 0.0003).
No change: If no change (power 1) were truly right, a best power this far from 1 would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001).
Method: R car::powerTransform(lm(time ~ poison + treatment), family = "bcPower"): the power by maximum likelihood with its Wald 95 % interval, the rounded power car suggests, and likelihood-ratio tests of power 0 and 1 (car::testTransform).
Note: for counts or shares, a model made for them (explain(..., outcome="counts" or "shares")) is 

,test,column,power,95 % CI low,95 % CI high,simple power,rows,p-value for power 0 (log),p-value for power 1 (no change)
1,best power (Box-Cox),time,-0.750162,-1.136931,-0.363394,-1.0,48,0.000299,4.918300e-14


In [13]:
dataset = "poisons (Box & Cox 1964)"
row = rows(poisons.find_best_power_of, "time", using=["poison", "treatment"])[0]
checks.check_values(dataset, "find_best_power_of (using=)",
                    {quantity: row[quantity] for quantity in ("power", "simple power", "95 % CI low", "95 % CI high")})
row = rows(poisons.find_best_power_of, "time")[0]
checks.check_values(dataset, "find_best_power_of",
                    {quantity: row[quantity] for quantity in ("power", "95 % CI low", "95 % CI high")})

agrees  poisons (Box & Cox 1964) | find_best_power_of (using=) | power: simplydata -0.75016230148, Box & Cox (1964), printed -0.75; difference 1.62e-04 (allowed 5e-03)
      Source: Box & Cox (1964), printed (printed) - https://doi.org/10.1111/j.2517-6161.1964.tb00553.x
agrees  poisons (Box & Cox 1964) | find_best_power_of (using=) | power: simplydata -0.75016230148, statsmodels OLS + SciPy, the profile (Wald interval) -0.750162514769; relative difference 2.84e-07 (allowed 1e-06)
      Source: statsmodels OLS + SciPy, the profile (Wald interval) (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=OLS
agrees  poisons (Box & Cox 1964) | find_best_power_of (using=) | power: simplydata -0.75016230148, R car powerTransform -0.75016230148; relative difference 0.00e+00 (allowed 1e-06)
      Source: R car powerTransform (run reference) - https://search.r-project.org/CRAN/refmans/car/html/powerTransform.html (version 3.1.5)
agrees  poisons (Box & Cox 1964) | find_best_power_of (

4

## Lines when both columns have errors, or with outliers: `table.fit_line(method=)`

- **Standardised major axis** of log10 leaf lifespan on log10 leaf mass per area (67 species; Wright et al., the
  smatr data `leaflife`, analysed by Warton et al. 2006): slope = sign(r) sd(y) / sd(x), intercept = mean(y) - slope
  mean(x) (closed formula); R `smatr::sma` for the interval.
- **Deming** line of creatinine in plasma on serum (108 patients, the mcr data) with y_error_over_x_error = 4 and 1:
  with lambda the ratio of the error variances, slope = (syy - lambda sxx + sqrt((syy - lambda sxx)^2 +
  4 lambda sxy^2)) / (2 sxy) (closed formula); R `mcr::mcreg(method.reg = "Deming", error.ratio = 1 / lambda)` and
  `deming::deming` for the standard errors.
- **Passing-Bablok** line on the same data: R `mcr::mcreg(method.reg = "PaBa")`.
- **Theil-Sen** line on Anscombe's set 3 and on the creatinine data: SciPy `theilslopes(method="separate")` (the
  parent) and R `deming::theilsen`.

In [14]:
leaflife = load("leaflife")
creatinine = load("creatinine")
anscombe = load("anscombe")
creatinine.fit_line(x="serum", y="plasma", method="deming", y_error_over_x_error=4)

"plasma" goes up by 1.02 for each 1 more "serum" (95 % CI 0.950 to 1.09), by the deming line.
Line: plasma = 1.02 * serum - 0.0141 (intercept 95 % CI -0.102 to 0.0740), over 108 rows.
This line answers another question than least squares: it compares two measurement methods that both carry error, with a known ratio of their error variances.
Method: R mcr::mcreg(x, y, method.reg = "Deming", error.ratio = 0.250, method.ci = "analytical"): the variance of the errors in "plasma" is taken as 4.00 times that in "serum" (y_error_over_x_error=4.00; mcr's error.ratio is x over y, so it gets 1 / 4.00). Intervals by mcr's analytical method.


,test,what it compares,slope,slope 95 % CI low,slope 95 % CI high,slope standard error,intercept,intercept 95 % CI low,intercept 95 % CI high,intercept standard error,p-value,R squared,rows,y error over x error
1,Deming line (R mcr),"""plasma"" against ""serum""",1.017863,0.950228,1.085498,0.034114,-0.014128,-0.102238,0.073982,0.044442,NaN,NaN,108,4.0


In [15]:
LINE = ["slope", "intercept", "slope standard error", "intercept standard error", "slope 95 % CI low",
        "slope 95 % CI high", "intercept 95 % CI low", "intercept 95 % CI high"]


def line_values(row):
    return {quantity: row[quantity] for quantity in LINE if quantity in row}


row = rows(leaflife.fit_line, x="log_lma", y="log_longev", method="standardised major axis")[0]
checks.check_values("leaflife (smatr)", 'fit_line (method="standardised major axis")', line_values(row))
for lam in (4, 1):
    row = rows(creatinine.fit_line, x="serum", y="plasma", method="deming", y_error_over_x_error=lam)[0]
    checks.check_values("creatinine (mcr)", f'fit_line (method="deming", y_error_over_x_error={lam})', line_values(row))
row = rows(creatinine.fit_line, x="serum", y="plasma", method="passing-bablok")[0]
checks.check_values("creatinine (mcr)", 'fit_line (method="passing-bablok")', line_values(row))
for dataset, data, x, y in (("Anscombe set 3", anscombe, "x3", "y3"), ("creatinine (mcr)", creatinine, "serum", "plasma")):
    row = rows(data.fit_line, x=x, y=y, method="theil-sen")[0]
    checks.check_values(dataset, 'fit_line (method="theil-sen")', line_values(row))

agrees  leaflife (smatr) | fit_line (method="standardised major axis") | slope: simplydata 1.31503061916, the formula 1.31503061916; relative difference 1.69e-16 (allowed 1e-10)
      Source: the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/14_small_additions.py
agrees  leaflife (smatr) | fit_line (method="standardised major axis") | slope: simplydata 1.31503061916, R smatr sma 1.31503061916; relative difference 1.69e-16 (allowed 1e-10)
      Source: R smatr sma (run reference) - https://search.r-project.org/CRAN/refmans/smatr/html/sma.html (version 3.5.2)
agrees  leaflife (smatr) | fit_line (method="standardised major axis") | intercept: simplydata -2.69880047585, the formula -2.69880047585; relative difference 0.00e+00 (allowed 1e-10)
      Source: the formula (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/14_small_additions.py
agrees  leaflife (smatr) | fit_line (method="sta

## The covariance table and MANOVA

`find_correlation_between(show="covariance")` on mtcars (fuel use, weight, horsepower): pandas `DataFrame.cov`
(the parent) and R `cov`. `compare_averages_of([...], for_each_group_in=)` with a list of columns is a one-way MANOVA:
on iris, sepal and petal length, and all four measurements, across species. Parent: statsmodels `MANOVA`; R:
`summary(manova(...), test = ...)` for Pillai's trace, Wilks' lambda, the Hotelling-Lawley trace and Roy's largest
root, with their approximate F and degrees of freedom.

In [16]:
mtcars = load("mtcars")
iris = load("iris")
iris.compare_averages_of(["Sepal.Length", "Petal.Length"], for_each_group_in="Species")

Comparing the averages of "Sepal.Length" and "Petal.Length" together for each "Species" (150 rows, 3 groups): a MANOVA, one test for all the columns at once.
  setosa: Sepal.Length 5.01, Petal.Length 1.46
  versicolor: Sepal.Length 5.94, Petal.Length 4.26
  virginica: Sepal.Length 6.59, Petal.Length 5.55
  Pillai's trace = 0.989, approximate F = 71.8, df = 4 and 294; fewer than 1 in 1000 studies would show this much if the averages truly did not differ in any of the columns (p < 0.0001).
Method: statsmodels MANOVA.from_formula("Sepal.Length + Petal.Length ~ Species").mv_test(); Pillai's trace, the one R summary(manova(cbind(Sepal.Length, Petal.Length) ~ Species)) prints by default and the safest when groups differ in size or spread; Wilks, Hotelling-Lawley and Roy are in the table that comes back.
To see which columns differ: compare_averages_of() for each column, then table.add_adjusted_p_value_column() on the p-values.


,term,statistic name,statistic,F,degrees of freedom 1,degrees of freedom 2,p-value
1,Species (Pillai's trace),Pillai's trace,0.988501,71.828795,4.0,294.000000,2.216888e-42
2,Species (Wilks' lambda),Wilks' lambda,0.039878,292.556466,4.0,292.000000,8.415775e-101
3,Species (Hotelling-Lawley trace),Hotelling-Lawley trace,23.364650,850.898575,4.0,174.165289,4.115997e-113
4,Species (Roy's greatest root),Roy's greatest root,23.334153,1715.060228,2.0,147.000000,1.297499e-102


In [17]:
covariance = keyed(mtcars.find_correlation_between, "column", "mpg", "wt", "hp", show="covariance")
COLUMNS = ["mpg", "wt", "hp"]
checks.check_values("mtcars", 'find_correlation_between (show="covariance")',
                    {f"{a} with {b}": covariance[a][b] for i, a in enumerate(COLUMNS) for b in COLUMNS[i:]})
TESTS = ["Pillai's trace", "Wilks' lambda", "Hotelling-Lawley trace", "Roy's greatest root"]
for columns in (["Sepal.Length", "Petal.Length"], ["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"]):
    found = keyed(iris.compare_averages_of, "statistic name", columns, for_each_group_in="Species")
    checks.check_values("iris", f"compare_averages_of (MANOVA of {len(columns)} columns)",
                        {f"{test}: {quantity}": found[test][quantity] for test in TESTS
                         for quantity in ("statistic", "F", "degrees of freedom 1", "degrees of freedom 2", "p-value")})

agrees  mtcars | find_correlation_between (show="covariance") | mpg with mpg: simplydata 36.3241028226, pandas DataFrame.cov 36.3241028226; relative difference 0.00e+00 (allowed 1e-10)
      Source: pandas DataFrame.cov (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=DataFrame.cov
agrees  mtcars | find_correlation_between (show="covariance") | mpg with mpg: simplydata 36.3241028226, R cov 36.3241028226; relative difference 0.00e+00 (allowed 1e-10)
      Source: R cov (run reference) - https://search.r-project.org/R/refmans/stats/html/cov.html
agrees  mtcars | find_correlation_between (show="covariance") | mpg with wt: simplydata -5.11668467742, pandas DataFrame.cov -5.11668467742; relative difference 1.74e-16 (allowed 1e-10)
      Source: pandas DataFrame.cov (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=DataFrame.cov
agrees  mtcars | find_correlation_between (show="covariance") | mpg with wt: simplydata -5.11668

agrees  iris | compare_averages_of (MANOVA of 2 columns) | Pillai's trace: statistic: simplydata 0.988500527043, statsmodels MANOVA 0.988500527043; relative difference 1.12e-16 (allowed 1e-10)
      Source: statsmodels MANOVA (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=MANOVA
agrees  iris | compare_averages_of (MANOVA of 2 columns) | Pillai's trace: statistic: simplydata 0.988500527043, R summary.manova 0.988500527043; relative difference 1.48e-14 (allowed 1e-10)
      Source: R summary.manova (run reference) - https://search.r-project.org/R/refmans/stats/html/summary.manova.html
agrees  iris | compare_averages_of (MANOVA of 2 columns) | Pillai's trace: F: simplydata 71.8287954469, statsmodels MANOVA 71.8287954469; relative difference 0.00e+00 (allowed 1e-10)
      Source: statsmodels MANOVA (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=MANOVA
agrees  iris | compare_averages_of (MANOVA of 2 columns) | Pillai's trace: F: simplydata 71.828795

## Result

In [18]:
results = checks.finish()

14_small_additions: 327 comparisons - 301 agrees, 11 explained difference – reference, 15 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
